# Email Finder — End-to-End Test Notebook (EF-19)

Pipeline:
1. Define leads inline
2. Qualify existing emails (discard hosting-platform / brand-mismatch emails)
3. Run `find_emails_batch` (waterfall + Mailin)
4. Inspect Perplexity responses
5. Export enriched + still_needs CSVs

## Cell 1: Setup & Config

In [ ]:
import sys, io, textwrap, logging
import pandas as pd
sys.path.append("../..")

from email_finder import (
    find_emails_batch, LeadInput, EmailFinderResult,
    load_leads_from_csv, load_leads_from_google_sheet, export_results,
)
from email_finder.config import Config
from email_finder.io.loader import _row_to_lead
from email_finder.io.qualify import qualify_lead_email, is_email_qualified

logging.basicConfig(level=logging.INFO)
config = Config()
print("Config loaded.")

## Cell 2: Define Test Leads (inline CSV)

`Podcast Name` → `full_name` (no host-name column in this sheet).  
Perplexity will find the host via podcast name + website.

In [ ]:
RAW_CSV = textwrap.dedent("""\
Podcast Name,Podcast Website,Podcast Email,Podcast Facebook,Podcast Twitter,Podcast Instagram,Podcast YouTube,Podcast LinkedIn
A Mommy And A Mic,http://www.amommyandamic.com/,podcast@myrockerbeez.com,,,,,
Culinary Treasure Podcast,https://www.culinarytreasurepodcast.com/,sshomler@me.com,https://www.facebook.com/CulinaryTreasurePodcast,,https://www.instagram.com/culinarytreasurepodcast,https://www.youtube.com/channel/UCA-zUuYpU_KQpVemaheeWEA,
Joy of Weightlessness,https://www.buzzsprout.com/2016334,,https://www.facebook.com/joalibng,https://twitter.com/JOALIBEING,https://www.instagram.com/joalibeing,,
TravelRight.Today,http://www.travelright.today/,,,,,,
Paper Trails,https://papertrails.podbean.com/,,,,,,
Waves of Impact,https://uwf.edu/commerce,,,,,,
The Smoking Barrel Podcast,http://thesmokingbarrelpodcast.com/,,https://www.facebook.com/thesmokingbarrelpodcast,,,,
The Southern Fork,http://www.thesouthernfork.com/episodes/,charlotteghost@gmail.com,,,"https://www.instagram.com/southernfork",,
Chef D's Bistro,https://podcasters.spotify.com/pod/show/darryl-ingram,darryl.ingram0162@gmail.com,,,,,
Grounded,https://www.groundedthepod.com/,"smoody09@gmail.com, tech@ringmaster.com",https://www.facebook.com/MichaelKLaRue,,https://www.instagram.com/tridavetri,,https://www.linkedin.com/in/amyhom17
""")

COLUMN_MAPPING = {
    "Podcast Name":      "full_name",
    "Podcast Website":   "website",
    "Podcast Email":     "existing_email",
    "Podcast Facebook":  "facebook_url",
    "Podcast Twitter":   "twitter_url",
    "Podcast Instagram": "instagram_url",
    "Podcast YouTube":   "youtube_url",
    "Podcast LinkedIn":  "linkedin_url",
}

df = pd.read_csv(io.StringIO(RAW_CSV), dtype=str, keep_default_na=False).replace("nan", "")
print(f"Parsed {len(df)} rows")
df

## Cell 3: Build LeadInput Objects

In [ ]:
raw_leads = []
for _, row in df.iterrows():
    lead = _row_to_lead(row, COLUMN_MAPPING)
    if lead:
        raw_leads.append(lead)

print(f"Built {len(raw_leads)} LeadInput objects:")
for l in raw_leads:
    print(f"  {l.full_name:<35}  email={l.existing_email or '—'}")

## Cell 4: Qualify Existing Emails

Same logic as the Google Apps Script `qualifyPodcastLeads()`:
- Discard emails whose domain is a podcast hosting platform
- Discard emails where LCS(brand candidates, email local/domain) < 4 chars

Discarded emails → `existing_email=None` → finder runs **Flow B** (full discovery) instead of **Flow A** (verify existing).

In [ ]:
leads = []
for lead in raw_leads:
    qualified_lead, reason = qualify_lead_email(lead)
    leads.append(qualified_lead)
    
    if reason:
        print(f"  [DISCARD] {lead.full_name}")
        print(f"            email: {lead.existing_email}")
        print(f"            reason: {reason}")
    elif lead.existing_email:
        print(f"  [KEEP]    {lead.full_name}")
        print(f"            email: {lead.existing_email}")
    else:
        print(f"  [NO EMAIL] {lead.full_name} → Flow B")

print(f"\n{sum(1 for l in leads if l.existing_email)} leads with qualified email (Flow A)")
print(f"{sum(1 for l in leads if not l.existing_email)} leads with no email (Flow B)")

## Cell 5: Run the Pipeline

In [ ]:
results = await find_emails_batch(leads, config)

## Cell 6: Preview Results

In [ ]:
status_icon = {"verified": "✓", "catch_all": "~", "unverified": "?", "not_found": "✗", "invalid": "✗"}

print(f"{'Name':<35}  {'Email':<40}  {'Status':<12}  Conf")
print("-" * 105)
for lead, result in zip(leads, results):
    icon = status_icon.get(result.status, "?")
    email_str = result.email or "N/A"
    conf = f"{result.confidence:.0%}" if result.confidence else ""
    print(f"[{icon}] {lead.full_name:<33}  {email_str:<40}  {result.status:<12}  {conf}")

## Cell 7: Inspect Perplexity Responses

Shows the exact prompt sent and raw text returned by Perplexity for each lead.

In [ ]:
for lead, result in zip(leads, results):
    perplexity_entries = [e for e in result.discovery_log if e.get("node") == "perplexity"]
    if not perplexity_entries:
        continue
    
    print(f"\n{'='*80}")
    print(f"LEAD: {lead.full_name}")
    
    for i, entry in enumerate(perplexity_entries, 1):
        res = entry.get("result", {})
        print(f"  [Perplexity call #{i}]")
        print(f"  PROMPT:")
        print(f"    {res.get('prompt', '—')}")
        print(f"  FOUND EMAIL: {res.get('found_email') or res.get('found_emails') or '—'}")
        print(f"  RAW RESPONSE:")
        raw = res.get('raw_response', '')
        # Print first 600 chars to keep output readable
        print(f"    {raw[:600]}{'...' if len(raw) > 600 else ''}")

## Cell 8: Debug — Deep Dive on One Lead

In [ ]:
idx = 0  # Change to inspect a different lead
lead = leads[idx]
result = results[idx]

print(f"Lead:       {lead.full_name}")
print(f"Email:      {result.email}")
print(f"Status:     {result.status}")
print(f"Confidence: {result.confidence:.0%}")
print(f"Source:     {result.source}")
print()
print("Discovery log:")
for entry in result.discovery_log:
    node = entry.get('node', '?')
    res = entry.get('result', {})
    found = res.get('found_email') or res.get('found_emails') or '—'
    err = res.get('error') or ''
    print(f"  [{node:<20}]  found={found}  {'ERROR: ' + err if err else ''}")

## Cell 9: Export to CSV

In [ ]:
files = export_results(leads, results, output_dir="./output", prefix="test_run")
print(f"\nEnriched:      {files['enriched']}")
print(f"Still needs:   {files['needs_enrichment']}")

## Cell 10: Quick Email Qualification Tester

Test the qualification logic against any email + podcast name without running the full pipeline.

In [ ]:
test_cases = [
    ("podcast@myrockerbeez.com",      "A Mommy And A Mic",          "http://www.amommyandamic.com/"),
    ("sshomler@me.com",               "Culinary Treasure Podcast",   "https://www.culinarytreasurepodcast.com/"),
    ("charlotteghost@gmail.com",      "The Southern Fork",           "http://www.thesouthernfork.com/"),
    ("smoody09@gmail.com",            "Grounded",                    "https://www.groundedthepod.com/"),
    ("darryl.ingram0162@gmail.com",   "Chef D's Bistro",             "https://podcasters.spotify.com/pod/show/darryl-ingram"),
]

print(f"{'Email':<40}  {'Podcast':<30}  {'Result':<8}  Reason")
print("-" * 110)
for email, name, site in test_cases:
    ok, reason = is_email_qualified(email, name, site)
    icon = "✓ KEEP" if ok else "✗ DROP"
    print(f"{email:<40}  {name:<30}  {icon:<8}  {reason}")